In [1]:
import pandas as pd
import altair as alt

In [2]:
crime = pd.read_csv("crime.csv").sort_values("MONTH", ascending=False)

In [3]:
# OCCURRED_ON_DATE is a string like "12/9/17 11:15"
crime["OCCURRED_ON_DATE"] = pd.to_datetime(
    crime["OCCURRED_ON_DATE"], format="%m/%d/%y %H:%M", errors="coerce"
)
crime = crime.dropna(subset=["OCCURRED_ON_DATE"])

# Keep one school year: Sep 1, 2017 - Aug 31, 2018
START, END = "2017-09-01", "2018-08-31"
crime = crime[crime["OCCURRED_ON_DATE"].between(START, f"{END} 23:59:59")].copy()

TYPE = "OFFENSE_CODE_GROUP"
crime["date"] = crime["OCCURRED_ON_DATE"].dt.floor("D")
crime["hour"] = crime["HOUR"].astype(int)
crime["dow"] = crime["DAY_OF_WEEK"].str.strip()

types = ["All"] + sorted(crime[TYPE].dropna().unique())

## Map 1: Zoomable incident map

One dot per incident, colored by offense, on a dark street map you can drag and zoom (scroll, or the + / − buttons; ⌂ returns to Northeastern).
Use the **Day** dropdown, the **Hours** range slider (drag either end to set the start and end hour), and the **Offense** checkboxes to filter.

In [4]:
import anywidget
import ipywidgets as widgets
import plotly.express as px
import requests
import traitlets

# One dot per offense row; drop rows without a usable location (blank or -1)
located = crime[crime["Lat"].between(42, 43) & crime["Long"].between(-72, -70)].copy()
located["STREET"] = located["STREET"].fillna("Unknown").str.title()
located["OFFENSE_DESCRIPTION"] = located["OFFENSE_DESCRIPTION"].str.capitalize()
located["when"] = located["OCCURRED_ON_DATE"].dt.strftime("%a, %b %d, %Y %I:%M %p")
print(f"{len(located):,} of {len(crime):,} rows have a location")

17,179 of 17,989 rows have a location


In [5]:
def count_grid(df, keys, levels):
    """Count incidents per type x keys, including an 'All' type and zero-count cells."""
    both = pd.concat([df.assign(type="All"), df.assign(type=df[TYPE])])
    # Unique incident numbers, so one incident logged with two offense codes
    # (e.g. I142025834-00) counts once
    counts = both.groupby(["type", *keys])["INCIDENT_NUMBER"].nunique()
    idx = pd.MultiIndex.from_product([types, *levels], names=["type", *keys])
    return counts.reindex(idx, fill_value=0).rename("incidents").reset_index()


days = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
week = count_grid(crime, ["dow", "hour"], [days, range(24)])

In [6]:
type_param = alt.param(
    name="crime_type",
    value="All",
    bind=alt.binding_select(options=types, name="Crime type: "),
)

# Day of week x hour heatmap (same as Viz 1)
weekly = (
    alt.Chart(week)
    .transform_filter(alt.datum.type == type_param)
    .mark_rect(stroke="white", strokeWidth=0.5)
    .encode(
        x=alt.X(
            "hour:O", title="Hour of day (0 = midnight)", axis=alt.Axis(labelAngle=0)
        ),
        y=alt.Y("dow:O", sort=days, title=None),
        color=alt.Color(
            "incidents:Q",
            scale=alt.Scale(scheme="purples"),
            legend=alt.Legend(title="Incidents"),
        ),
        tooltip=[
            alt.Tooltip("dow:N", title="Day"),
            alt.Tooltip("hour:O", title="Hour"),
            alt.Tooltip("incidents:Q", title="Incidents"),
        ],
    )
    .add_params(type_param)
    .properties(
        width=540,
        height=180,
        title="Incidents by day of week and hour, Sep 2017 - Aug 2018",
    )
)
weekly

alt.Chart(...)

In [7]:
# Zoomable map: a small custom widget that draws the map with MapLibre (the engine
# Plotly's maps use), so the zoom buttons, pin, and filters all run in the browser.
# Needs a running kernel and internet; a saved/exported notebook won't show the map.
offenses = types[1:]
colors = dict(zip(offenses, px.colors.qualitative.T10))
# Red is reserved for the Northeastern pin; white stands out on the dark map
colors["Commercial Burglary"] = "#FFFFFF"

# Northeastern University (Centennial Common) and the map's starting zoom
NEU_LAT, NEU_LON = 42.3398, -71.0892
START_ZOOM = 13

MAP_JS = r"""
import maplibregl from "https://esm.sh/maplibre-gl@4.7.1";

const STYLE = "https://basemaps.cartocdn.com/gl/dark-matter-gl-style/style.json";

function render({ model, el }) {
  // MapLibre's stylesheet, inserted next to the map (VS Code ignores the widget's _css)
  const style = document.createElement("style");
  style.textContent = model.get("maplibre_css");
  el.appendChild(style);

  const box = document.createElement("div");
  // Fill whatever width the notebook gives the map (up to `width`), so nothing is cut off
  box.style.width = "100%";
  box.style.maxWidth = model.get("width") + "px";
  box.style.height = model.get("height") + "px";
  el.appendChild(box);

  const home = model.get("home");
  const zoom = model.get("start_zoom");
  const data = model.get("data");
  const map = new maplibregl.Map({ container: box, style: STYLE, center: home, zoom });

  // + / - zoom buttons and a home button that flies back to Northeastern
  // (plain text buttons with their styles set right here, so they show in any notebook)
  const controls = document.createElement("div");
  controls.style.cssText =
    "position:absolute;top:10px;left:10px;z-index:10;display:flex;flex-direction:column;gap:4px;";
  for (const [text, title, action] of [
    ["+", "Zoom in", () => map.zoomIn()],
    ["−", "Zoom out", () => map.zoomOut()],
    ["⌂", "Back to Northeastern", () => map.flyTo({ center: home, zoom })],
  ]) {
    const button = document.createElement("button");
    button.type = "button";
    button.title = title;
    button.textContent = text;
    button.style.cssText =
      "width:32px;height:32px;padding:0;cursor:pointer;background:#fff;color:#222;" +
      "border:1px solid #bbb;border-radius:6px;font:600 20px/30px sans-serif;" +
      "box-shadow:0 1px 4px rgba(0,0,0,0.4);";
    button.onmouseenter = () => (button.style.background = "#eee");
    button.onmouseleave = () => (button.style.background = "#fff");
    button.onclick = action;
    controls.appendChild(button);
  }
  box.style.position = "relative";
  box.appendChild(controls);

  // Northeastern: red teardrop pin plus a label to its right
  const pin = new maplibregl.Marker({ color: "#EA4335" }).setLngLat(home).addTo(map);
  pin.getElement().title = "Northeastern University";
  const label = document.createElement("div");
  label.textContent = "Northeastern";
  label.style.cssText = "color:white;font:600 13px sans-serif;text-shadow:0 0 3px #000,0 0 3px #000;";
  new maplibregl.Marker({ element: label, anchor: "left", offset: [12, -20] }).setLngLat(home).addTo(map);

  const when = (t) => new Date(Date.UTC(2017, 8, 1) + t * 60000).toLocaleString("en-US", {
    timeZone: "UTC", weekday: "short", month: "short", day: "2-digit", year: "numeric",
    hour: "2-digit", minute: "2-digit",
  });

  function applyFilter() {
    if (!map.getLayer("dots")) return;
    const f = model.get("filter");
    const rules = ["all",
      ["in", ["get", "o"], ["literal", f.offenses]],
      [">=", ["get", "h"], f.start],
      ["<=", ["get", "h"], f.end]];
    if (f.day >= 0) rules.push(["==", ["get", "w"], f.day]);
    map.setFilter("dots", rules);
  }

  map.on("load", () => {
    const features = data.lon.map((lon, i) => ({
      type: "Feature",
      geometry: { type: "Point", coordinates: [lon, data.lat[i]] },
      properties: { i, o: data.o[i], w: data.w[i], h: data.h[i] },
    }));
    map.addSource("incidents", { type: "geojson", data: { type: "FeatureCollection", features } });
    map.addLayer({
      id: "dots",
      type: "circle",
      source: "incidents",
      paint: {
        "circle-radius": ["interpolate", ["linear"], ["zoom"], 10, 2.5, 15, 6],
        "circle-color": ["match", ["get", "o"], ...data.colors.flatMap((c, k) => [k, c]), "#999"],
        "circle-opacity": 0.85,
      },
    });
    applyFilter();

    // Hover a dot to see the offense, description, time, and street
    const popup = new maplibregl.Popup({ closeButton: false, closeOnClick: false });
    map.on("mousemove", "dots", (e) => {
      map.getCanvas().style.cursor = "pointer";
      const i = e.features[0].properties.i;
      popup.setLngLat(e.features[0].geometry.coordinates)
        .setHTML(`<div style="color:#222;font:12px sans-serif"><b>${data.offenses[data.o[i]]}</b>` +
                 `<br>${data.descriptions[data.d[i]]}<br>${when(data.t[i])}<br>${data.streets[data.s[i]]}</div>`)
        .addTo(map);
    });
    map.on("mouseleave", "dots", () => {
      map.getCanvas().style.cursor = "";
      popup.remove();
    });
  });

  model.on("change:filter", applyFilter);

  // Redraw at the right size if the notebook output area changes width
  const resizer = new ResizeObserver(() => map.resize());
  resizer.observe(box);
  return () => {
    resizer.disconnect();
    map.remove();
  };
}

export default { render };
"""

# MapLibre's stylesheet, fetched here and passed to the map, which inserts it itself
MAPLIBRE_CSS = requests.get(
    "https://unpkg.com/maplibre-gl@4.7.1/dist/maplibre-gl.css", timeout=20
).text


class IncidentMap(anywidget.AnyWidget):
    _esm = MAP_JS
    maplibre_css = traitlets.Unicode(MAPLIBRE_CSS).tag(sync=True)
    data = traitlets.Dict().tag(sync=True)  # all points, sent once
    filter = traitlets.Dict().tag(sync=True)  # which points to show
    home = traitlets.List([NEU_LON, NEU_LAT]).tag(sync=True)
    start_zoom = traitlets.Float(START_ZOOM).tag(sync=True)
    width = traitlets.Int(820).tag(sync=True)
    height = traitlets.Int(600).tag(sync=True)


# Send every point once as compact columns (text as codes into lookup lists);
# the filters then just tell the map which points to show
offense_codes = located[TYPE].map({o: k for k, o in enumerate(offenses)})
desc_codes, descriptions = pd.factorize(located["OFFENSE_DESCRIPTION"], sort=True)
street_codes, streets = pd.factorize(located["STREET"], sort=True)
incident_map = IncidentMap(
    # Let the map take the space left over next to the checkboxes
    layout=widgets.Layout(flex="1 1 auto", min_width="0"),
    data=dict(
        lon=located["Long"].round(5).tolist(),
        lat=located["Lat"].round(5).tolist(),
        o=offense_codes.tolist(),
        d=desc_codes.tolist(),
        s=street_codes.tolist(),
        w=located["dow"].map({d: k for k, d in enumerate(days)}).tolist(),
        h=located["hour"].tolist(),
        t=((located["OCCURRED_ON_DATE"] - pd.Timestamp(START)).dt.total_seconds() // 60)
        .astype(int)
        .tolist(),
        offenses=offenses,
        descriptions=list(descriptions),
        streets=list(streets),
        colors=[colors[o] for o in offenses],
    ),
)

# Title and incident count above the map
title = widgets.HTML(
    '<div style="font-size:18px"><b>Where Incidents Happened</b>, Sep 2017 - Aug 2018</div>'
)
shown = widgets.HTML()

day = widgets.Dropdown(options=["All days", *days], description="Day:")
# Two-handle slider: start and end hour, both included (0-23 = all hours)
hour = widgets.IntRangeSlider(
    value=(0, 23),
    min=0,
    max=23,
    description="Hours:",
    continuous_update=False,
    layout=widgets.Layout(width="500px"),
)

# Offense checkboxes, each with its map color, plus All / None buttons
checks = {
    offense: widgets.Checkbox(value=True, description=offense, indent=False)
    for offense in offenses
}
select_all = widgets.Button(description="All", layout=widgets.Layout(width="70px"))
select_none = widgets.Button(description="None", layout=widgets.Layout(width="70px"))


def set_all(value):
    # Pause the per-box updates so the map updates once instead of nine times
    for box in checks.values():
        box.unobserve(update, "value")
        box.value = value
        box.observe(update, "value")
    update()


select_all.on_click(lambda _: set_all(True))
select_none.on_click(lambda _: set_all(False))

offense_panel = widgets.VBox(
    [widgets.HTML("<b>Offense</b>")]
    + [
        widgets.HBox(
            [
                widgets.HTML(
                    f'<span style="display:inline-block;width:12px;height:12px;border-radius:50%;'
                    f'background:{colors[o]};border:1px solid #888"></span>'
                ),
                checks[o],
            ],
            layout=widgets.Layout(align_items="center"),
        )
        for o in offenses
    ]
    + [widgets.HBox([select_all, select_none])],
    layout=widgets.Layout(margin="40px 0 0 12px", flex="0 0 auto"),
)


def update(_=None):
    picked = [o for o, box in checks.items() if box.value]
    start, end = hour.value
    sub = located[located[TYPE].isin(picked) & located["hour"].between(start, end)]
    if day.value != "All days":
        sub = sub[sub["dow"] == day.value]
    incident_map.filter = dict(
        offenses=[offenses.index(o) for o in picked],
        day=-1 if day.value == "All days" else days.index(day.value),
        start=start,
        end=end,
    )
    shown.value = f"{len(sub):,} incidents shown"


day.observe(update, "value")
hour.observe(update, "value")
for box in checks.values():
    box.observe(update, "value")
update()

widgets.VBox(
    [
        title,
        shown,
        widgets.HBox([day, hour]),
        widgets.HBox([incident_map, offense_panel]),
    ]
)